# Marilou — Chronos-2 Pendle benchmark (NOT EXECUTED)

Notebook préparé sans dataset. Aucun résultat pré-rempli. Modèle zero-shot,
sans entraînement. Choisir GPU dans Colab si disponible ; CPU accepté.

Importer le code local de `marilou` dans `/content/bsa-hackathon` (archive ou
copie du dépôt). Cette branche n’est pas encore publiée. Importer séparément
les JSON Data V1. Les chemins ci-dessous sont des paramètres, pas des données
existantes. Le rapport reste **RESEARCH_ONLY**, hors cache de décision.

Sources : [modèle officiel](https://huggingface.co/amazon/chronos-2),
[API officielle](https://github.com/amazon-science/chronos-forecasting).


In [ ]:
from pathlib import Path
REPO_ROOT = Path('/content/bsa-hackathon')
SERIES_PATH = Path('/content/data/pt_series.json')
STRATEGY_PATH = Path('/content/data/strategy.json')
SNAPSHOT_REF = ''  # Référence stable/hash du snapshot réel
SOURCE_AS_OF = ''  # ISO 8601 avec timezone, fourni par Data
RETRIEVED_AT = ''  # ISO 8601 avec timezone, fourni par Data
MODEL_REVISION = ''  # Commit SHA de amazon/chronos-2 (40 caractères)
DEVICE = 'auto'  # GPU si disponible, sinon CPU
REPORT_PATH = Path('/content/marilou_research_report.json')


In [ ]:
import subprocess
import sys
if not (REPO_ROOT / 'ml/chronos2.py').is_file():
    raise FileNotFoundError('Importer d’abord le code marilou à REPO_ROOT')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r',
                       str(REPO_ROOT / 'ml/requirements-colab.txt')])
sys.path.insert(0, str(REPO_ROOT))


In [ ]:
from datetime import datetime
from backend.app.models.timeseries import TimeseriesDTO
from backend.app.models.strategy import StrategyDTO
from ml.series import prepare_pt_series

for path in (SERIES_PATH, STRATEGY_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'Dataset absent : {path}')
if not all((SNAPSHOT_REF, SOURCE_AS_OF, RETRIEVED_AT)):
    raise ValueError('Métadonnées de collecte nécessaires avant le benchmark')
series_dto = TimeseriesDTO.model_validate_json(SERIES_PATH.read_text())
strategy_dto = StrategyDTO.model_validate_json(STRATEGY_PATH.read_text())
series = prepare_pt_series(
    series_dto, strategy_dto, source_snapshot_ref=SNAPSHOT_REF,
    source_as_of=datetime.fromisoformat(SOURCE_AS_OF.replace('Z', '+00:00')),
    retrieved_at=datetime.fromisoformat(RETRIEVED_AT.replace('Z', '+00:00')),
)
print({'data_mode': series.data_mode.value, 'observations': len(series.values),
       'cutoff': series.timestamps[-1], 'expiry': series.expiry,
       'entity_id': series.entity_id})


In [ ]:
import pandas as pd
history = pd.Series([float(v) for v in series.values], index=series.timestamps)
display(history.describe())
history.plot(title='PT spot USD — source history', ylabel='USD per PT')


In [ ]:
from dataclasses import asdict
from ml.benchmark import backtest, persistence, rolling_mean
print('Naive:', [asdict(r) for r in backtest(series, persistence)])
print('Rolling mean:', [asdict(r) for r in backtest(series, rolling_mean)])


## Chargement explicite et mesure du temps

La cellule suivante télécharge les poids et charge le modèle. Ne la lancer
qu’après l’audit de la source. Copier un commit de la page Hugging Face dans
MODEL_REVISION pour une exécution reproductible. Les imports précédents ne
chargent aucun modèle. La disponibilité des GPU Colab varie.


In [ ]:
from time import perf_counter
import torch
from ml.chronos2 import Chronos2Forecaster, evaluate
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
started = perf_counter()
model = Chronos2Forecaster.load(revision=MODEL_REVISION, device=DEVICE, seed=42)
load_seconds = perf_counter() - started
print('Download + load seconds:', load_seconds)


In [ ]:
started = perf_counter()
report = evaluate(series, model)
report['total_evaluation_seconds'] = perf_counter() - started
report['model_load_seconds'] = load_seconds
from importlib.metadata import version
report['environment'] = {name: version(name) for name in
                         ('chronos-forecasting', 'torch', 'pandas', 'pydantic')}
report['seed'] = 42
report['device'] = str(model.pipeline.model.device)
display(pd.DataFrame(report['horizon_metrics']))
print('Total evaluation seconds:', report['total_evaluation_seconds'])


In [ ]:
import json
REPORT_PATH.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('Research report saved:', REPORT_PATH)
# Download manually from Colab files; do not put this report in ForecastCache.


## Revue avant intégration

Comparer chaque horizon aux deux baselines, lire pinball loss et couverture,
vérifier maturité, fraîcheur et absence de révision/fuite temporelle. Le rapport
n’attribue jamais VALIDATED automatiquement. DEMO reste une fixture. Une
mauvaise performance ne doit pas être présentée comme un gain prédictif.

Le ForecastBundle contractuel sera créé après audit sur le dataset réel.
Le mode Normal et le prix oracle Morpho ne sont jamais modifiés par ce notebook.
